# N2 Master Proxy Lab v2 (Faz A: stabilizasyon & assert'li test)

`00_N2_Master_Proxy_Lab.ipynb` deneyinin tekrarlanabilir hali:
- Proxy kaynağı **repoda** (`../proxy/src/n2proxy_master.cpp`); notebook yalnızca *sync → build → run → assert* yapar.
- SSH yardımcıları `n2lab_ops.py` (MOTD yok, `pkill -x`, parola `VM_PASS` env).
- Senaryolar `chaos_suite.py`: her biri **PASS / FAIL / XFAIL** (XFAIL = ilgili Faz B maddesiyle bilinen açık).
- UE → gNB eşlemesi doğru: s1,s3→gNB-A · s1sd1→gNB-B · s2→gNB-C (v1'de yalnız gNB-A açılıyordu).


## 1. Bağlantı ve hazırlık

In [1]:
import importlib, n2lab_ops as ops, chaos_suite as cs
importlib.reload(ops); importlib.reload(cs)
print("VM:", "✅" if ops.vm_ok() else "❌ erişilemiyor")
ops.kill_ran(); print(ops.ensure_amfs()[1])
print(ops.ensure_proxy_profiles())

VM: ✅
▶ gNB/UE/proxy temizliği
gNB=0 UE=0 proxy=0
amf: active
amf2: active
amf3: active
gnb-a-proxy.yaml ok
gnb-b-proxy.yaml ok
gnb-c-proxy.yaml ok


## 2. Proxy: senkronla → derle → başlat

In [2]:
ops.sync_proxy() and ops.build_proxy() and ops.start_proxy()

✅ sync src/n2proxy_master.cpp
✅ sync src/ngap.h
✅ sync src/lb_core.h
✅ derlendi
✅ proxy çalışıyor


True

## 3. Senaryolar
`baseline` (ardışık) ve `concurrent` (eşzamanlı) 3 gNB + 4 UE kaydı; `stop/kill9/blackhole/sigstop` arıza modları; `amf_loss_reset` (NG Reset) ve `reconnect` (yeniden bağlanma + NGSetup replay).

In [3]:
cs.main(['baseline', 'concurrent'])


━━ baseline ━━
✅ sync src/n2proxy_master.cpp
✅ sync src/ngap.h
✅ sync src/lb_core.h
✅ derlendi
✅ proxy çalışıyor
   ✔ gNB-a NG Setup (proxy üzerinden)
   ✔ gNB-b NG Setup (proxy üzerinden)
   ✔ gNB-c NG Setup (proxy üzerinden)
   ✔ UE-s1 (gNB-a) PDU session
   ✔ UE-s1sd1 (gNB-b) PDU session
   ✔ UE-s2 (gNB-c) PDU session
   ✔ UE-s3 (gNB-a) PDU session
   ✔ 4 UE → 4 [LB] olayı
   ✔ 3 AMF'in üçü de en az 1 UE aldı
   ℹ  farklı gNB'lerde aynı RAN_ID (B1 çakışma riski): {1: {9, 13, 5}}

━━ concurrent ━━
✅ sync src/n2proxy_master.cpp
✅ sync src/ngap.h
✅ sync src/lb_core.h
✅ derlendi
✅ proxy çalışıyor
   ✔ gNB-a NG Setup
   ✔ gNB-b NG Setup
   ✔ gNB-c NG Setup
   ✔ UE-s1 (gNB-a) PDU session
   ✔ UE-s1sd1 (gNB-b) PDU session
   ✔ UE-s2 (gNB-c) PDU session
   ✔ UE-s3 (gNB-a) PDU session
   ✔ 4 UE → 4 [LB] olayı
   ✔ bilinmeyen UE / yanlış AMF uyarısı yok

══ SONUÇ ══
  baseline   PASS   
  concurrent PASS   


0

In [4]:
cs.main(['stop', 'kill9', 'blackhole'])


━━ stop ━━
✅ sync src/n2proxy_master.cpp
✅ sync src/ngap.h
✅ sync src/lb_core.h
✅ derlendi
✅ proxy çalışıyor
   ✔ gNB-a NG Setup
   ✔ UE-s1 (öncesi) PDU session
   ⚡ arıza: stop
   ✔ proxy AMF-2 kaybını algıladı ([HA] pasife)
   ✔ arıza sonrası UE-s3 PDU session
   ✔ UE-s3 için yeni [LB] olayı var ve ölü AMF-2'ye atanmadı

━━ kill9 ━━
✅ sync src/n2proxy_master.cpp
✅ sync src/ngap.h
✅ sync src/lb_core.h
✅ derlendi
✅ proxy çalışıyor
   ✔ gNB-a NG Setup
   ✔ UE-s1 (öncesi) PDU session
   ⚡ arıza: kill9
   ✔ proxy AMF-2 kaybını algıladı ([HA] pasife)
   ✔ arıza sonrası UE-s3 PDU session
   ℹ  AMF-2 test sırasında yeniden havuza döndü (systemd Restart + proxy reconnect)
   ✔ UE-s3 için yeni [LB] olayı var
   ✔ AMF-2 systemd ile döndü ve proxy havuza aldı (reconnect + NGSetup replay)

━━ blackhole ━━
✅ sync src/n2proxy_master.cpp
✅ sync src/ngap.h
✅ sync src/lb_core.h
✅ derlendi
✅ proxy çalışıyor
   ✔ gNB-a NG Setup
   ✔ UE-s1 (öncesi) PDU session
   ⚡ arıza: blackhole
   ✔ proxy AMF-2 kayb

0

In [5]:
cs.main(['amf_loss_reset', 'reconnect'])


━━ amf_loss_reset ━━
✅ sync src/n2proxy_master.cpp
✅ sync src/ngap.h
✅ sync src/lb_core.h
✅ derlendi
✅ proxy çalışıyor
   ✔ gNB-a NG Setup
   ✔ UE-s1/UE-s3 PDU session
   ✔ UE'lerden biri AMF-2'de
   ℹ  kayıp sonrası proxy olayları ([LB]/[HA]/[WARN]; S-TMSI affinity görünüyor mu? release neden iki kez?):
      05:08:32.748 [HA] gNB (FD=5) icin 1 UE'ye UEContextReleaseCommand gonderildi
      05:08:32.748 [HA] AMF'i kaybolan UE (RAN_ID=2) release tamamlandi
      05:08:32.950 [LB] YENI UE (GNB_FD=5 RAN_ID=3) -> AMF-3 secildi! (S-TMSI sahibi AMF yok, Agirlikli RR)
      05:08:33.153 [LB] YENI UE (GNB_FD=5 RAN_ID=4) -> AMF-1 secildi! (Agirlikli RR)
      05:08:35.800 [HA] AMF-2 kapasite=100 GUAMI=1 adet (ilk: set=1 ptr=1)
      05:08:35.800 [HA] AMF-2 NGSetup tamam — havuzda (READY)
   ✔ AMF-2 kaybı + etkilenen UE=1
   ✔ etkilenen UE için UEContextReleaseCommand gönderildi
   ✔ gNB release'i tamamladı (ReleaseComplete)
   ℹ  gNB-a log (release/reset/unhandled/error):
      [2026-10-12 05

0

In [6]:
cs.main(['weights', 'drain'])   # Faz C: kapasite-ağırlıklı LB, GUAMI, drain


━━ weights ━━
✅ sync src/n2proxy_master.cpp
✅ sync src/ngap.h
✅ sync src/lb_core.h
✅ derlendi
✅ proxy çalışıyor
   ✔ gNB-a NG Setup
   ✔ kapasiteler öğrenildi: AMF-1=100, AMF-2=100, AMF-3=50
   ℹ  GUAMI: [('2', '1', '1', '1'), ('1', '1', '1', '0'), ('3', '1', '1', '2')]   (affinity için her AMF'in pointer'ı benzersiz olmalı: 0/1/2)
   ✔ her AMF'in en az 1 GUAMI'si ve benzersiz pointer'ı var
   ✔ UE-s1/UE-s3 PDU session
   ✔ seçim ağırlıklı RR olarak loglandı

━━ drain ━━
✅ sync src/n2proxy_master.cpp
✅ sync src/ngap.h
✅ sync src/lb_core.h
✅ derlendi
✅ proxy çalışıyor
   ✔ gNB-a NG Setup
   ✔ AMF-1 drain'e alındı
   ✔ UE-s1/UE-s3 PDU session
   ✔ drain'deki AMF-1'e UE atanmadı
   ✔ AMF-1 drain'den çıkarıldı
   ℹ  durum çıktısı:
      05:09:40.497 [CTL] AMF-1 drain moduna alindi (yeni UE almaz)
      05:09:40.497 [CTL] durum: gNB=1 UE=0
      05:09:40.497 [CTL]   AMF-1 127.0.0.5:38412 durum=READY kapasite=100 drain=evet
      05:09:40.497 [CTL]   AMF-2 127.0.0.105:38412 durum=READY kapa

0

### Faz D: yük testi + metrikler
Sentetik gNB'ler (`load_gen.py`, aynı RAN id'ler) proxy'ye eşzamanlı InitialUE yağdırır; yanıt kaybı, dağılım, gecikme, proxy CPU/RSS ve Prometheus sayaçları raporlanır. `load` = 3×50 @100/sn (yanıt kaybı yok şart), `load_mid` = 3×100 @200/sn, `load_big` = 3×200 @400/sn (yüksek hızda darboğaz genellikle 5GC zinciridir: proxy kontrolleri zorunlu, 5GC yanıt oranı bilgilendirici; CPU dağılımı raporlanır). Proxy mesaj korunumu (`rx == tx + yutulan + düşen`) her koşuda doğrulanır.

In [7]:
cs.main(['load', 'load_mid', 'load_big'])


━━ load ━━
✅ sync src/n2proxy_master.cpp
✅ sync src/ngap.h
✅ sync src/lb_core.h
✅ derlendi
✅ proxy çalışıyor
✅ sync load_gen.py + load_templates.py → /home/localadmin/loadgen
   ✔ yük üreteci VM'e kopyalandı
   ℹ  /etc/open5gs/scp.yaml (yorumsuz):
      logger:
        file:
          path: /var/log/open5gs/scp.log
      global:
        max:
          ue: 1024  # The number of UE can be increased depending on memory size.
      scp:
        sbi:
          server:
            - address: 127.0.0.200
              port: 7777
          client:
            nrf:
              - uri: http://127.0.0.10:7777
   ℹ  150 gönderildi / 150 yanıt · gönderim 100.6/sn · gecikme ms p50=20.04 p95=30.26 p99=30.51 max=30.62
   ℹ  CPU, GÖNDERİM PENCERESİNDE (çekirdek; 4 vCPU): open5gs-scpd=0.094, open5gs-udmd=0.047, open5gs-ausfd=0.034, open5gs-amfd=0.034, mongod=0.027, open5gs-udrd=0.02, n2proxy_master=0.013  · proxy RSS=1704 kB
   ℹ  toplam CPU saniyesi (tüm koşu): open5gs-scpd=0.14, open5gs-udmd=0.07, o

0

### Yanıtsızlık teşhisi: kontrol deneyi + hız taraması\n`load_direct` aynı yükü proxy'siz doğrudan AMF-1'e verir (kayıp orada da varsa proxy masum). `load_sweep` hızı artırarak yanıt oranının düştüğü eşiği ve SCP 'Connection timer expired' sayısını raporlar (uzun sürer: ~8-10 dk).

In [8]:
cs.main(['load_direct', 'load_sweep'])


━━ load_direct ━━
✅ sync src/n2proxy_master.cpp
✅ sync src/ngap.h
✅ sync src/lb_core.h
✅ derlendi
✅ proxy çalışıyor
✅ sync load_gen.py + load_templates.py → /home/localadmin/loadgen
   ✔ yük üreteci VM'e kopyalandı
   ℹ  DOĞRUDAN AMF-1: 300 gönderildi / 262 yanıt (87%) · gönderim 300.9/sn · gecikme ms p50=6.7 p95=10.43 max=13.61
   ℹ  CPU (gönderim penceresi): open5gs-scpd=0.231, open5gs-amfd=0.11, open5gs-udmd=0.09, open5gs-ausfd=0.09, open5gs-udrd=0.06, mongod=0.05
   ℹ  open5gs-scpd: 45 hata/uyarı — 18× [scp] WARNING: response_handler() failed [-N] (../src/scp/sbi-path.c:N) | 18× [sbi] ERROR: Connection timer expired [METHOD:POST] (../lib/sbi/client.c:N)
   ℹ  open5gs-amfd@amf: 1197 hata/uyarı — 597× [amf] WARNING: [suci-N-N-N-N-N-N-N]    RAN_UE_NGAP_ID[N] AMF_UE_NGAP_ID[N] (../src/amf/amf | 300× [amf] WARNING:     RAN_UE_NGAP_ID[N] AMF_UE_NGAP_ID[N] (../src/amf/amf-sm.c:N)
   ⚠ proxy'siz doğrudan AMF-1'e 300 UE'nin hepsi yanıtlandı  [262/300 — kayıp proxy'siz de var ⇒ proxy masum,

0

In [9]:
cs.main(['sigstop'])   # XFAIL beklenir: kernel SCTP donmuş süreci algılatmaz


━━ sigstop ━━
✅ sync src/n2proxy_master.cpp
✅ sync src/ngap.h
✅ sync src/lb_core.h
✅ derlendi
✅ proxy çalışıyor
   ✔ gNB-a NG Setup
   ✔ UE-s1 (öncesi) PDU session
   ⚡ arıza: sigstop
   ✘ proxy AMF-2 kaybını algıladı ([HA] pasife)
   ✘ arıza sonrası UE-s3 PDU session  [82] [rrc] [info] UE switches to state [RRC-IDLE] | [2026-10-12 05:18:38.583] [nas] [info] UE switches to state [CM-IDLE]]
   ℹ  AMF-2 test sırasında yeniden havuza döndü (systemd Restart + proxy reconnect)
   ✔ UE-s3 için yeni [LB] olayı var

══ SONUÇ ══
  sigstop    XFAIL  Kernel SCTP donmuş (SIGSTOP) sürecin yerine heartbeat/ACK verir; tespit için uygulama düzeyi sağlık yoklaması gerekir (Faz C4/yeni madde)


0

### Uygulama düzeyi yoklama (opt-in, `N2P_PROBE_MS`)
Önce `probe_healthy` PASS olmalı: sağlıklı Open5GS AMF'in boş RANConfigurationUpdate'e yanıt verdiğini kanıtlar. Aksi halde yoklama açmayın.

In [10]:
cs.main(['probe_healthy'])


━━ probe_healthy ━━
✅ sync src/n2proxy_master.cpp
✅ sync src/ngap.h
✅ sync src/lb_core.h
✅ derlendi
✅ proxy çalışıyor
   ✔ gNB-a NG Setup
   ✔ 15 sn boyunca hiçbir AMF yoklamada düşmedi
   ✔ yoklama açıkken UE-s1/UE-s3 PDU session

══ SONUÇ ══
  probe_healthy PASS   


0

In [11]:
cs.main(['sigstop_probe'])   # probe_healthy PASS ise: donmuş AMF algılanmalı


━━ sigstop_probe ━━
✅ sync src/n2proxy_master.cpp
✅ sync src/ngap.h
✅ sync src/lb_core.h
✅ derlendi
✅ proxy çalışıyor
   ✔ gNB-a NG Setup
   ✔ UE-s1 (öncesi) PDU session
   ⚡ arıza: sigstop
   ✔ proxy AMF-2 kaybını algıladı ([HA] pasife)
   ✔ arıza sonrası UE-s3 PDU session
   ✔ UE-s3 için yeni [LB] olayı var ve ölü AMF-2'ye atanmadı

══ SONUÇ ══
  sigstop_probe PASS   


0

## 4. Proxy logu

In [12]:
print(ops.proxy_log())

05:19:32.268 [PROXY] Master Load Balancer devrede... (IP: 127.0.0.50, PORT: 38412, AMF sayisi: 3, AMF kaybinda: UE release, HB: 1000 ms, probe: 1500 ms)
05:19:34.334 [PROXY] Yeni gNB bağlandı! (FD=5) AMF'lere (Havuz) bağlanılıyor...
05:19:34.337   -> AMF-1 (127.0.0.5) BAĞLANDI (FD=6)
05:19:34.337   -> AMF-2 (127.0.0.105) BAĞLANDI (FD=7)
05:19:34.337   -> AMF-3 (127.0.0.205) BAĞLANDI (FD=8)
05:19:34.338 [HA] AMF-1 kapasite=100 GUAMI=1 adet (ilk: set=1 ptr=0)
05:19:34.338 [HA] AMF-1 NGSetup tamam — havuzda (READY)
05:19:34.338 [HA] AMF-2 kapasite=100 GUAMI=1 adet (ilk: set=1 ptr=1)
05:19:34.338 [HA] AMF-2 NGSetup tamam — havuzda (READY)
05:19:34.338   -> AMF-2 yaniti (proc=21) gNB'ye zaten iletilmisti — yutuldu
05:19:34.338 [HA] AMF-3 kapasite=50 GUAMI=1 adet (ilk: set=1 ptr=2)
05:19:34.338 [HA] AMF-3 NGSetup tamam — havuzda (READY)
05:19:34.338   -> AMF-3 yaniti (proc=21) gNB'ye zaten iletilmisti — yutuldu
05:19:35.399 [LB] YENI UE (GNB_FD=5 RAN_ID=1) -> AMF-1 secildi! (Agirlikli RR)
05

## 5. Temizlik

In [13]:
ops.kill_ran(); print(ops.ensure_amfs()[1])

▶ gNB/UE/proxy temizliği
gNB=0 UE=0 proxy=0
amf: active
amf2: active
amf3: active
